In [0]:
%pip install xgboost scikit-learn

In [0]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pandas as pd
import numpy as np
import json

from sklearn.preprocessing import LabelEncoder
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, TimestampType, DateType
)
from delta.tables import DeltaTable
from datetime import date

In [0]:
dbutils.widgets.text("job_parameters", "{}")

parameters = json.loads(dbutils.widgets.get("job_parameters"))

print(f"Raw widget value : {repr(dbutils.widgets.get('job_parameters'))}")
print(f"Parsed keys      : {list(parameters.keys())}")

# ── Identity & routing ────────────────────────────────────────────────────
catalog          = parameters.get("catalog")
_model_name      = parameters.get("model_name")
_target_table    = parameters.get("target_table")
_registry_table  = parameters.get("registry_table", "gold.model_registry")
primary_keys     = parameters.get("primary_keys")
job_id           = parameters.get("job_id")
parent_job       = parameters.get("parent_job_id", job_id)
partition        = parameters.get("partition")

# ── Inference type ────────────────────────────────────────────────────────
# "classification" → predict_proba + threshold + probability tiers
# "regression"     → predict + inverse transform + value tiers
prediction_type  = parameters.get("prediction_type", "classification")

# ── Shared inference params ───────────────────────────────────────────────
passthrough_cols = parameters.get("passthrough_cols", ["customer_state", "customer_city"])
fill_with_zero   = parameters.get("fill_with_zero", [])
inference_experiment_path = parameters.get(
    "inference_experiment_path",
    f"/Workspace/Users/sahil.prusty09@gmail.com/experiments/ecommerce_inference"
)

# ── Classification-specific params ───────────────────────────────────────
# Tier offset from best_threshold — read from registry
# e.g. best_threshold=0.45, high_offset=0.25 → high risk >= 0.70
high_offset = float(parameters.get("high_offset", 0.25))
tier_labels = parameters.get("tier_labels", {
    "high": "High", "medium": "Medium", "low": "Low"
})

# ── Regression-specific params ────────────────────────────────────────────
# Whether to inverse log-transform predictions back to original scale
log_transform_target = parameters.get("log_transform_target", False)

# BRL value thresholds for CLV tier assignment
# high   → predicted_clv >= high_value_cutoff
# medium → predicted_clv >= medium_value_cutoff
# low    → below medium_value_cutoff
clv_tier_cutoffs = parameters.get("clv_tier_cutoffs", {
    "high"  : 50.0,    # >= 50 BRL in 90 days → High value
    "medium": 15.0,    # >= 15 BRL in 90 days → Medium value
})
clv_tier_labels = parameters.get("clv_tier_labels", {
    "high"  : "High Value",
    "medium": "Medium Value",
    "low"   : "Low Value"
})

# ── Validate required params ──────────────────────────────────────────────
required = {
    "catalog"        : catalog,
    "model_name"     : _model_name,
    "target_table"   : _target_table,
    "primary_keys"   : primary_keys,
    "job_id"         : job_id,
    "partition"      : partition,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(
        f"\n❌ Missing required parameters: {missing}\n"
        f"   Parsed values: { {k: parameters.get(k) for k in missing} }"
    )

if prediction_type not in ("classification", "regression"):
    raise ValueError(
        f"❌ Invalid prediction_type: '{prediction_type}'\n"
        f"   Must be 'classification' or 'regression'"
    )

# ── Construct full table names ────────────────────────────────────────────
model_name     = _model_name
target_table   = f"{catalog}.{_target_table}"
registry_table = f"{catalog}.{_registry_table}"

print("=" * 60)
print("  INFERENCE PARAMETERS LOADED")
print("=" * 60)
print(f"  Catalog             : {catalog}")
print(f"  Prediction type     : {prediction_type}")
print(f"  Model name          : {model_name}")
print(f"  Target table        : {target_table}")
print(f"  Registry table      : {registry_table}")
print(f"  Passthrough cols    : {passthrough_cols}")
print(f"  Job ID              : {job_id}")
print(f"  Partition           : {partition}")
if prediction_type == "regression":
    print(f"  Log transform       : {log_transform_target}")
    print(f"  CLV tier cutoffs    : {clv_tier_cutoffs}")
else:
    print(f"  High offset         : best_threshold + {high_offset}")
    print(f"  Tier labels         : {tier_labels}")
print("=" * 60)

In [0]:
# ── Read production entry for this model ─────────────────────────────────
registry_df = spark.table(registry_table) \
    .filter(F.col("model_name") == model_name) \
    .filter(F.col("status") == "production") \
    .orderBy(F.col("updated_at").desc()) \
    .limit(1)

if registry_df.count() == 0:
    raise RuntimeError(
        f"\n❌ No production model found for '{model_name}' in {registry_table}\n"
        f"   Possible causes:\n"
        f"   1. Model name is wrong — check exact name in registry table\n"
        f"   2. Model status is not 'production'\n"
        f"   3. Training notebook has not been run yet\n\n"
        f"   Available models:\n"
        f"{spark.table(registry_table).select('model_name','status','updated_at').show()}"
    )

registry_row = registry_df.toPandas().iloc[0]

# ── Extract all inference config from registry ────────────────────────────
model_uri       = registry_row["model_uri"]
model_version   = registry_row["mlflow_version"]
source_view     = registry_row["source_view"]
target_col      = registry_row["target_col"]
feature_cols    = json.loads(registry_row["feature_cols"])
best_threshold  = float(registry_row["best_threshold"])
score_col       = registry_row["score_col"]
label_col       = registry_row["label_col"]
train_auc       = float(registry_row["auc_roc"])
train_f1        = float(registry_row["f1_score"])

# ── Derive categorical cols from feature_cols ────────────────────────────
# Any col ending in _encoded was a categorical — reverse to get originals
categorical_cols = [
    col.replace("_encoded", "")
    for col in feature_cols
    if col.endswith("_encoded")
]

print("=" * 55)
print("  MODEL CONFIG LOADED FROM REGISTRY")
print("=" * 55)
print(f"  model_uri      : {model_uri}")
print(f"  model_version  : {model_version}")
print(f"  source_view    : {source_view}")
print(f"  target_col     : {target_col}")
print(f"  feature_cols   : {len(feature_cols)} features")
print(f"  best_threshold : {best_threshold}")
print(f"  score_col      : {score_col}")
print(f"  label_col      : {label_col}")
print(f"  training AUC   : {train_auc}")
print(f"  training F1    : {train_f1}")
print(f"  categorical    : {categorical_cols}")
print("=" * 55)

In [0]:
import mlflow
import mlflow.xgboost
import mlflow.sklearn

print(f"Loading model      : {model_uri}")
print(f"Prediction type    : {prediction_type}")

# ── Detect model framework from model_uri artifact name ──────────────────
# "clv_random_forest"    → sklearn
# "clv_xgboost_regressor"→ xgboost
# "xgboost_tuned"        → xgboost (classification)
artifact_name = model_uri.split("/")[-1].lower()
use_sklearn   = "random_forest" in artifact_name or "sklearn" in artifact_name

try:
    if use_sklearn:
        model      = mlflow.sklearn.load_model(model_uri)
        model_framework = "sklearn"
    else:
        model      = mlflow.xgboost.load_model(model_uri)
        model_framework = "xgboost"

    print(f"✅ Model loaded successfully")
    print(f"   Framework : {model_framework}")
    print(f"   Type      : {type(model).__name__}")

    # Feature count — works for both sklearn and xgboost
    n_features = (
        model.n_features_in_
        if hasattr(model, "n_features_in_")
        else model.n_features_in_
    )
    print(f"   Features  : {n_features}")

except Exception as e:
    raise RuntimeError(
        f"\n❌ Failed to load model\n"
        f"   model_uri  : {model_uri}\n"
        f"   Framework  : {'sklearn' if use_sklearn else 'xgboost'}\n"
        f"   Error      : {str(e)}\n\n"
        f"   Fix: check model_uri in registry matches the artifact logged during training"
    )

In [0]:
# ── Load ALL customers — no null filter unlike training ───────────────────
df_all = spark.table(source_view).toPandas()

print(f"✅ Loaded {source_view}")
print(f"   Total rows : {df_all.shape[0]:,}")
print(f"   Columns    : {df_all.shape[1]}")

# ── Validate feature + primary + passthrough columns exist ────────────────
raw_feature_cols = [c for c in feature_cols if not c.endswith("_encoded")]
all_required     = raw_feature_cols + primary_keys + passthrough_cols
missing_cols     = [c for c in all_required if c not in df_all.columns]

if missing_cols:
    raise ValueError(
        f"❌ Missing columns in source view: {missing_cols}\n"
        f"   Check 'passthrough_cols' in job JSON — "
        f"only include columns that exist in {source_view}"
    )
print(f"✅ All expected columns present")

# ── Keep primary keys + passthrough cols for output ───────────────────────
df_keys = df_all[primary_keys + passthrough_cols].copy()

# ── Encode categorical columns ────────────────────────────────────────────
encoders               = {}
inference_feature_cols = [c for c in feature_cols if not c.endswith("_encoded")]

for col in categorical_cols:
    if col in df_all.columns:
        le           = LabelEncoder()
        encoded_col  = f"{col}_encoded"
        df_all[encoded_col] = le.fit_transform(df_all[col].astype(str))
        encoders[col]       = le
        if col in inference_feature_cols:
            inference_feature_cols.remove(col)
        inference_feature_cols.append(encoded_col)
        print(f"✅ Encoded {col} → {encoded_col} ({df_all[encoded_col].nunique()} categories)")
    else:
        print(f"⚠️  Skipped {col} — not found in dataframe")

# ── Fill nulls using parameterized fill_with_zero list ───────────────────
zero_cols   = [c for c in fill_with_zero   if c in inference_feature_cols]
median_cols = [c for c in inference_feature_cols if c not in fill_with_zero]

df_all[zero_cols]   = df_all[zero_cols].fillna(0)
df_all[median_cols] = df_all[median_cols].fillna(
    df_all[median_cols].median(numeric_only=True)
)

print(f"\n✅ Zero-filled   : {len(zero_cols)} columns")
print(f"✅ Median-filled : {len(median_cols)} columns")

# ── Cast to float ─────────────────────────────────────────────────────────
X_all = df_all[inference_feature_cols].astype(float)

# ── Validate feature count matches model ─────────────────────────────────
if X_all.shape[1] != model.n_features_in_:
    raise ValueError(
        f"\n❌ FEATURE MISMATCH\n"
        f"   Model expects : {model.n_features_in_} features\n"
        f"   Data has      : {X_all.shape[1]} features\n"
        f"   Difference    : {set(feature_cols) - set(inference_feature_cols)}"
    )

print(f"\n✅ Inference matrix    : {X_all.shape}")
print(f"   Model expects      : {model.n_features_in_} features ✅")
print(f"   Null values remain : {X_all.isna().sum().sum()}")

In [0]:
import numpy as np

print(f"Scoring {X_all.shape[0]:,} customers...")
print(f"Prediction type : {prediction_type}")

if prediction_type == "classification":

    raw_scores   = model.predict_proba(X_all)[:, 1]
    pred_labels  = (raw_scores >= best_threshold).astype(int)

    high_cutoff   = best_threshold + high_offset
    medium_cutoff = best_threshold

    def assign_tier(prob):
        if prob >= high_cutoff:      return tier_labels["high"]
        elif prob >= medium_cutoff:  return tier_labels["medium"]
        else:                        return tier_labels["low"]

    pred_tiers    = [assign_tier(p) for p in raw_scores]
    tier_col_name = "risk_tier"

    n_positive    = pred_labels.sum()
    tier_counts   = pd.Series(pred_tiers).value_counts()
    high_count    = tier_counts.get(tier_labels["high"],   0)
    medium_count  = tier_counts.get(tier_labels["medium"], 0)
    low_count     = tier_counts.get(tier_labels["low"],    0)

    print(f"\n{'='*60}")
    print(f"  CLASSIFICATION SCORING COMPLETE")
    print(f"{'='*60}")
    print(f"  Total scored            : {len(raw_scores):,}")
    print(f"  Predicted positive (=1) : {n_positive:,}  ({n_positive/len(raw_scores)*100:.1f}%)")
    print(f"  Predicted negative (=0) : {len(raw_scores)-n_positive:,}  ({(len(raw_scores)-n_positive)/len(raw_scores)*100:.1f}%)")
    print(f"  Threshold used          : {best_threshold}")
    print(f"{'='*60}")
    print(f"  Tier Breakdown:")
    print(f"  {tier_labels['high']:<12} (>= {high_cutoff:.2f})       : {high_count:,}  ({high_count/len(raw_scores)*100:.1f}%)")
    print(f"  {tier_labels['medium']:<12} ({medium_cutoff:.2f}–{high_cutoff:.2f})  : {medium_count:,}  ({medium_count/len(raw_scores)*100:.1f}%)")
    print(f"  {tier_labels['low']:<12} (<  {medium_cutoff:.2f})       : {low_count:,}  ({low_count/len(raw_scores)*100:.1f}%)")
    print(f"{'='*60}")
    print(f"\n  Probability distribution:")
    print(f"  Mean   : {raw_scores.mean():.4f}")
    print(f"  Median : {np.median(raw_scores):.4f}")
    print(f"  p90    : {np.percentile(raw_scores, 90):.4f}")

elif prediction_type == "regression":

    raw_scores_log = model.predict(X_all)

    if log_transform_target:
        raw_scores = np.expm1(np.clip(raw_scores_log, 0, None))
        print(f"✅ Applied expm1 inverse transform")
    else:
        raw_scores = np.clip(raw_scores_log, 0, None)

    high_cutoff   = float(clv_tier_cutoffs.get("high",   50.0))
    medium_cutoff = float(clv_tier_cutoffs.get("medium", 15.0))

    def assign_clv_tier(value):
        if value >= high_cutoff:     return clv_tier_labels["high"]
        elif value >= medium_cutoff: return clv_tier_labels["medium"]
        else:                        return clv_tier_labels["low"]

    pred_tiers    = [assign_clv_tier(v) for v in raw_scores]
    tier_col_name = "clv_tier"

    # ── FIX: For regression label_col = score_col (numeric BRL value) ────
    # pred_labels is the rounded numeric prediction — NOT the tier string
    pred_labels   = raw_scores.round(2)

    tier_counts   = pd.Series(pred_tiers).value_counts()
    high_count    = tier_counts.get(clv_tier_labels["high"],   0)
    medium_count  = tier_counts.get(clv_tier_labels["medium"], 0)
    low_count     = tier_counts.get(clv_tier_labels["low"],    0)

    print(f"\n{'='*60}")
    print(f"  REGRESSION SCORING COMPLETE")
    print(f"{'='*60}")
    print(f"  Total scored            : {len(raw_scores):,}")
    print(f"  Mean predicted CLV      : {raw_scores.mean():.2f} BRL")
    print(f"  Median predicted CLV    : {np.median(raw_scores):.2f} BRL")
    print(f"  Max predicted CLV       : {raw_scores.max():.2f} BRL")
    print(f"{'='*60}")
    print(f"  CLV Tier Breakdown:")
    print(f"  {clv_tier_labels['high']:<15} (>= {high_cutoff:.0f} BRL)  : {high_count:,}  ({high_count/len(raw_scores)*100:.1f}%)")
    print(f"  {clv_tier_labels['medium']:<15} ({medium_cutoff:.0f}–{high_cutoff:.0f} BRL) : {medium_count:,}  ({medium_count/len(raw_scores)*100:.1f}%)")
    print(f"  {clv_tier_labels['low']:<15} (<  {medium_cutoff:.0f} BRL)  : {low_count:,}  ({low_count/len(raw_scores)*100:.1f}%)")
    print(f"{'='*60}")
    print(f"\n  Prediction distribution (BRL):")
    print(f"  Mean   : {raw_scores.mean():.2f}")
    print(f"  Median : {np.median(raw_scores):.2f}")
    print(f"  p25    : {np.percentile(raw_scores, 25):.2f}")
    print(f"  p75    : {np.percentile(raw_scores, 75):.2f}")
    print(f"  p90    : {np.percentile(raw_scores, 90):.2f}")

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType
)

# ── Build output pandas DataFrame ────────────────────────────────────────
predictions_pd = df_keys.copy()

if prediction_type == "classification":
    predictions_pd[score_col]        = raw_scores.round(6)   # probability
    predictions_pd[label_col]        = pred_labels.astype(int) # 0 or 1
    predictions_pd[tier_col_name]    = pred_tiers             # High/Medium/Low
    predictions_pd["best_threshold"] = best_threshold
    predictions_pd["high_cutoff"]    = best_threshold + high_offset

elif prediction_type == "regression":
    # ── FIX: score_col = numeric prediction, tier_col = string tier ──────
    # label_col from registry = "clv_tier" but we use score_col for numeric
    # and tier_col_name for the string — no column name collision
    predictions_pd[score_col]        = raw_scores.round(2)   # BRL value
    predictions_pd[tier_col_name]    = pred_tiers             # High/Medium/Low Value
    predictions_pd["log_transformed"]= str(log_transform_target)

# ── Common columns ────────────────────────────────────────────────────────
predictions_pd["model_name"]      = model_name
predictions_pd["model_version"]   = str(model_version)
predictions_pd["prediction_date"] = str(date.today())
predictions_pd["job_id"]          = str(job_id)
predictions_pd["partition"]       = str(partition)

total = len(predictions_pd)

# ── Build schema dynamically ──────────────────────────────────────────────
schema_fields = []

# Primary keys
for pk in primary_keys:
    schema_fields.append(StructField(pk, StringType(), nullable=False))

# Passthrough cols
for col in passthrough_cols:
    schema_fields.append(StructField(col, StringType(), nullable=True))

# ── FIX: type-aware score and tier fields ─────────────────────────────────
if prediction_type == "classification":
    schema_fields.append(StructField(score_col,      DoubleType(),  nullable=False))  # probability
    schema_fields.append(StructField(label_col,      IntegerType(), nullable=False))  # 0 or 1
    schema_fields.append(StructField(tier_col_name,  StringType(),  nullable=False))  # risk tier
    schema_fields.append(StructField("best_threshold", DoubleType(), nullable=True))
    schema_fields.append(StructField("high_cutoff",    DoubleType(), nullable=True))

elif prediction_type == "regression":
    # score_col = numeric BRL prediction (double)
    # tier_col_name = string tier label (string)
    # NO label_col added — avoids collision since label_col == tier_col_name in registry
    schema_fields.append(StructField(score_col,      DoubleType(),  nullable=False))  # BRL value
    schema_fields.append(StructField(tier_col_name,  StringType(),  nullable=False))  # value tier
    schema_fields.append(StructField("log_transformed", StringType(), nullable=True))

# Common metadata fields
schema_fields += [
    StructField("model_name",      StringType(), nullable=False),
    StructField("model_version",   StringType(), nullable=True),
    StructField("prediction_date", StringType(), nullable=False),
    StructField("job_id",          StringType(), nullable=True),
    StructField("partition",       StringType(), nullable=True),
]

output_schema = StructType(schema_fields)
output_cols   = [f.name for f in schema_fields]

# ── Validate all output cols exist in predictions_pd ─────────────────────
missing_output_cols = [c for c in output_cols if c not in predictions_pd.columns]
if missing_output_cols:
    raise ValueError(
        f"❌ Missing output columns in predictions_pd: {missing_output_cols}\n"
        f"   Available: {list(predictions_pd.columns)}"
    )

predictions_spark = spark.createDataFrame(
    predictions_pd[output_cols],
    schema=output_schema
)

print(f"✅ Spark DataFrame created")
print(f"   Rows   : {predictions_spark.count():,}")
print(f"   Schema :")
predictions_spark.printSchema()

In [0]:
# ── CREATE TABLE — dynamic schema based on prediction_type ────────────────
if prediction_type == "classification":
    create_ddl = f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            customer_unique_id  STRING   NOT NULL,
            customer_state      STRING,
            customer_city       STRING,
            {score_col}         DOUBLE,
            {label_col}         INT,
            risk_tier           STRING,
            model_name          STRING,
            model_version       STRING,
            best_threshold      DOUBLE,
            high_cutoff         DOUBLE,
            prediction_date     STRING,
            job_id              STRING,
            partition           STRING
        )
        USING DELTA
        COMMENT 'Daily churn predictions — batch inference pipeline'
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """
elif prediction_type == "regression":
    create_ddl = f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            customer_unique_id  STRING   NOT NULL,
            customer_state      STRING,
            customer_city       STRING,
            {score_col}         DOUBLE,
            clv_tier            STRING,
            log_transformed     STRING,
            model_name          STRING,
            model_version       STRING,
            prediction_date     STRING,
            job_id              STRING,
            partition           STRING
        )
        USING DELTA
        COMMENT 'Daily CLV predictions — batch inference pipeline'
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """

spark.sql(create_ddl)
print(f"✅ Target table ready : {target_table}")

# ── Build MERGE set dynamically from actual source columns ────────────────
# Only include columns that BOTH exist in source DataFrame AND target table
source_cols  = set(predictions_spark.columns)
target_cols  = set(spark.table(target_table).columns)

# Columns never updated in MERGE — only used in join condition
exclude_from_update = {"customer_unique_id", "prediction_date"}

# Build update set — only cols present in both source and target
merge_set = {
    col: f"s.{col}"
    for col in source_cols
    if col not in exclude_from_update
    and col in target_cols
}

# ── Validate merge set is not empty ───────────────────────────────────────
if not merge_set:
    raise ValueError(
        f"❌ MERGE set is empty — no matching columns between source and target\n"
        f"   Source cols : {sorted(source_cols)}\n"
        f"   Target cols : {sorted(target_cols)}\n"
        f"   Fix: ensure predictions_spark has columns matching the target table"
    )

print(f"✅ MERGE set built — updating {len(merge_set)} columns:")
for col, expr in sorted(merge_set.items()):
    print(f"   {col:<30} ← {expr}")

# ── MERGE — upsert by customer + prediction_date ──────────────────────────
merge_condition = """
    t.customer_unique_id = s.customer_unique_id
    AND t.prediction_date = s.prediction_date
"""

DeltaTable.forName(spark, target_table) \
    .alias("t") \
    .merge(predictions_spark.alias("s"), merge_condition) \
    .whenMatchedUpdate(set=merge_set) \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"\n✅ MERGE complete → {target_table}")

In [0]:
# ── Read back what was written ────────────────────────────────────────────
written       = spark.table(target_table) \
    .filter(F.col("prediction_date") == str(date.today()))

written_count = written.count()
written_cols  = set(written.columns)

print(f"✅ Rows written today   : {written_count:,}")
print(f"   Columns in table    : {sorted(written_cols)}")

# ── Determine tier column based on prediction_type ────────────────────────
actual_tier_col = "risk_tier" if prediction_type == "classification" else "clv_tier"

checks_passed = True

# ── Check 1: row count matches input ─────────────────────────────────────
if written_count != total:
    print(f"⚠️  Check 1 failed — scored {total:,} but wrote {written_count:,}")
    checks_passed = False
else:
    print(f"✅ Check 1 passed — row count matches ({written_count:,})")

# ── Check 2: no nulls in score column ────────────────────────────────────
null_scores = written.filter(F.col(score_col).isNull()).count()
if null_scores > 0:
    print(f"⚠️  Check 2 failed — {null_scores} null scores found in '{score_col}'")
    checks_passed = False
else:
    print(f"✅ Check 2 passed — zero null scores in '{score_col}'")

# ── Check 3: score range validation (type-aware) ──────────────────────────
if prediction_type == "classification":
    # Churn probability must be between 0 and 1
    invalid_scores = written.filter(
        (F.col(score_col) < 0) | (F.col(score_col) > 1)
    ).count()
    if invalid_scores > 0:
        print(f"⚠️  Check 3 failed — {invalid_scores} probabilities outside [0, 1]")
        checks_passed = False
    else:
        print(f"✅ Check 3 passed — all probabilities in [0, 1]")

elif prediction_type == "regression":
    # CLV predictions must be non-negative BRL values
    invalid_scores = written.filter(F.col(score_col) < 0).count()
    if invalid_scores > 0:
        print(f"⚠️  Check 3 failed — {invalid_scores} negative CLV predictions found")
        checks_passed = False
    else:
        clv_stats = written.agg(
            F.min(score_col).alias("min"),
            F.avg(score_col).alias("mean"),
            F.max(score_col).alias("max")
        ).collect()[0]
        print(f"✅ Check 3 passed — all CLV predictions >= 0")
        print(f"   Min: {clv_stats['min']:.2f} BRL | "
              f"Mean: {clv_stats['mean']:.2f} BRL | "
              f"Max: {clv_stats['max']:.2f} BRL")

# ── Check 4: all 3 tiers present ─────────────────────────────────────────
if actual_tier_col in written_cols:
    tiers_present = [
        r[actual_tier_col]
        for r in written.select(actual_tier_col).distinct().collect()
    ]
    if len(tiers_present) < 3:
        print(f"⚠️  Check 4 warning — only {len(tiers_present)} tier(s) found: {tiers_present}")
    else:
        print(f"✅ Check 4 passed — all 3 tiers present: {sorted(tiers_present)}")
else:
    print(f"⚠️  Check 4 skipped — tier column '{actual_tier_col}' not in table")

# ── Check 5: tier distribution is reasonable ─────────────────────────────
if prediction_type == "classification":
    high_pct = high_count / total * 100
    if high_pct > 50:
        print(f"⚠️  Check 5 warning — {high_pct:.1f}% flagged as High risk")
        print(f"   Consider raising best_threshold in model_registry")
    else:
        print(f"✅ Check 5 passed — {high_pct:.1f}% High risk (acceptable)")

elif prediction_type == "regression":
    high_pct = high_count / total * 100
    low_pct  = low_count  / total * 100
    if high_pct > 80:
        print(f"⚠️  Check 5 warning — {high_pct:.1f}% flagged as High Value")
        print(f"   Consider raising clv_tier_cutoffs.high in job JSON")
    elif low_pct > 95:
        print(f"⚠️  Check 5 warning — {low_pct:.1f}% flagged as Low Value")
        print(f"   Consider lowering clv_tier_cutoffs in job JSON")
    else:
        print(f"✅ Check 5 passed — tier distribution looks reasonable")
        print(f"   High: {high_count:,} ({high_pct:.1f}%) | "
              f"Medium: {medium_count:,} ({medium_count/total*100:.1f}%) | "
              f"Low: {low_count:,} ({low_count/total*100:.1f}%)")

print(f"\n{'='*55}")
print(f"  OUTPUT VALIDATION : {'✅ ALL PASSED' if checks_passed else '⚠️  WARNINGS FOUND'}")
print(f"{'='*55}")

# ── Preview top customers (type-aware) ────────────────────────────────────
if prediction_type == "classification":
    preview_cols = [
        c for c in ["customer_unique_id", "customer_state",
                    score_col, label_col, "risk_tier",
                    "model_name", "prediction_date"]
        if c in written_cols
    ]
    print(f"\n📋 Top 10 highest churn probability customers:")
    display(
        written
        .orderBy(F.col(score_col).desc())
        .select(*preview_cols)
        .limit(10)
    )

elif prediction_type == "regression":
    preview_cols = [
        c for c in ["customer_unique_id", "customer_state",
                    score_col, "clv_tier",
                    "model_name", "prediction_date"]
        if c in written_cols
    ]
    print(f"\n📋 Top 10 highest predicted CLV customers:")
    display(
        written
        .orderBy(F.col(score_col).desc())
        .select(*preview_cols)
        .limit(10)
    )

In [0]:
spark.sql(f"OPTIMIZE {target_table} ZORDER BY (customer_unique_id, prediction_date)")
print(f"✅ OPTIMIZE complete : {target_table}")

try:
    mlflow.set_experiment(inference_experiment_path)

    with mlflow.start_run(run_name=f"inference_{prediction_type}_{job_id}"):
        mlflow.set_tags({
            "run_type"       : f"inference_{prediction_type}",
            "job_id"         : job_id,
            "parent_job"     : str(parent_job),
            "partition"      : str(partition),
            "model_name"     : model_name,
            "model_version"  : str(model_version),
            "prediction_date": str(date.today()),
            "target_table"   : target_table,
            "prediction_type": prediction_type,
        })

        if prediction_type == "classification":
            mlflow.log_metrics({
                "total_scored"     : float(total),
                "predicted_positive": float(n_positive),
                "positive_rate"    : round(float(n_positive) / total, 4),
                "high_risk_count"  : float(high_count),
                "medium_risk_count": float(medium_count),
                "low_risk_count"   : float(low_count),
                "mean_probability" : round(float(raw_scores.mean()), 4),
                "p90_probability"  : round(float(np.percentile(raw_scores, 90)), 4),
            })
            mlflow.log_params({
                "best_threshold" : best_threshold,
                "high_cutoff"    : best_threshold + high_offset,
            })

        elif prediction_type == "regression":
            mlflow.log_metrics({
                "total_scored"        : float(total),
                "mean_predicted_clv"  : round(float(raw_scores.mean()), 2),
                "median_predicted_clv": round(float(np.median(raw_scores)), 2),
                "p90_predicted_clv"   : round(float(np.percentile(raw_scores, 90)), 2),
                "high_value_count"    : float(high_count),
                "medium_value_count"  : float(medium_count),
                "low_value_count"     : float(low_count),
            })
            mlflow.log_params({
                "log_transform_target": log_transform_target,
                "clv_high_cutoff"     : clv_tier_cutoffs.get("high"),
                "clv_medium_cutoff"   : clv_tier_cutoffs.get("medium"),
            })

        mlflow.log_params({
            "source_model_uri": model_uri,
            "target_table"    : target_table,
            "prediction_type" : prediction_type,
        })

    print(f"✅ Inference run logged → {inference_experiment_path}")

except Exception as e:
    print(f"⚠️  MLflow logging skipped: {str(e)}")

print(f"\n{'='*60}")
print(f"  🎉 BATCH INFERENCE COMPLETE")
print(f"{'='*60}")
print(f"  Prediction type  : {prediction_type}")
print(f"  Customers scored : {total:,}")
print(f"  Target table     : {target_table}")
print(f"  Prediction date  : {date.today()}")
print(f"  Model            : {model_name} v{model_version}")
print(f"{'='*60}")